# Resume-to-Job-Description Matcher — Siamese LSTM

Loads the `resume_jd_pairs.csv` built in `ResumeJD_Logistic_Regression.ipynb` and trains a
**Siamese LSTM** network: both the resume and the job description are encoded through
the *same* shared embedding + LSTM branch, then the two encodings are combined and
passed to a small classifier head that outputs match probability.

In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Lambda, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


## 1. Load the pairs dataset

In [ ]:
pairs = pd.read_csv("resume_jd_pairs.csv")
print("Shape:", pairs.shape)
pairs.head(3)


In [ ]:
pairs.duplicated().sum()


In [ ]:
pairs.drop_duplicates(inplace=True)
pairs.dropna(inplace=True)


## 2. Train/test split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    pairs[["resume", "jd"]], pairs["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=pairs["label"]
)


## 3. Prepare Sequences for the Neural Network

### Tokenization
A single shared tokenizer is fit on **both** resumes and JDs from the training set, so the
resume branch and JD branch of the Siamese network draw from the same vocabulary.

### Padding
Resumes are longer documents than job descriptions, so we use separate max lengths:
`MAX_LEN_RESUME=300`, `MAX_LEN_JD=150`.

In [ ]:
MAX_LEN_RESUME = 300
MAX_LEN_JD = 150
VOCAB_SIZE = 15000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(pd.concat([X_train_text["resume"], X_train_text["jd"]]))

def encode(df):
    resume_seq = tokenizer.texts_to_sequences(df["resume"])
    jd_seq = tokenizer.texts_to_sequences(df["jd"])
    resume_pad = pad_sequences(resume_seq, maxlen=MAX_LEN_RESUME, padding="post", truncating="post")
    jd_pad = pad_sequences(jd_seq, maxlen=MAX_LEN_JD, padding="post", truncating="post")
    return resume_pad, jd_pad

X_train_resume, X_train_jd = encode(X_train_text)
X_test_resume, X_test_jd = encode(X_test_text)

y_train_arr = y_train.values
y_test_arr = y_test.values


## 4. Build the Siamese LSTM model

Both branches share the **same** `Embedding` + `LSTM` layers (true Siamese weight
sharing), so the network learns one text encoder used for both resumes and job descriptions.
The two encoded vectors are combined via absolute difference *and* concatenation before the
classifier head — the same merge strategy used for the QQP duplicate-question Siamese
networks.

In [ ]:
embedding_dim = 64
rnn_units = 64

# Shared layers (weight-tied across both branches)
shared_embedding = Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, mask_zero=True)
shared_rnn = LSTM(rnn_units)

resume_input = Input(shape=(MAX_LEN_RESUME,), name="resume_input")
jd_input = Input(shape=(MAX_LEN_JD,), name="jd_input")

resume_encoded = shared_rnn(shared_embedding(resume_input))
jd_encoded = shared_rnn(shared_embedding(jd_input))

abs_diff = Lambda(lambda x: tf.abs(x[0] - x[1]))([resume_encoded, jd_encoded])
merged = Concatenate()([resume_encoded, jd_encoded, abs_diff])

x = Dense(64, activation="relu")(merged)
x = Dropout(0.3)(x)
x = Dense(32, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

lstm_model = Model(inputs=[resume_input, jd_input], outputs=output)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    [X_train_resume, X_train_jd],
    y_train_arr,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)


## 5. Evaluate on the test set

In [ ]:
y_pred_lstm = (lstm_model.predict([X_test_resume, X_test_jd]) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred_lstm))


In [ ]:
accuracy_score(y_test_arr, y_pred_lstm)


In [ ]:
cm = confusion_matrix(y_test_arr, y_pred_lstm)
ConfusionMatrixDisplay(cm, display_labels=["No Match", "Match"]).plot()
plt.title("Siamese LSTM — Confusion Matrix")
plt.show()


In [ ]:
plt.plot(history_lstm.history["accuracy"], label="train acc")
plt.plot(history_lstm.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Siamese LSTM — Training curve")
plt.show()


## 6. Notes

Compare this LSTM's accuracy/F1 against `ResumeJD_Logistic_Regression.ipynb` and the
other RNN variants. The best-performing model (typically GRU) is the one saved and deployed
in `ResumeJD_GRU.ipynb` / `app.py`.